In [1]:
!apt-get -qq -y install fonts-nanum > /dev/null

import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import matplotlib as mpl

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
fm.fontManager.addfont(font_path)
plt.rc("font", family="NanumGothic")
plt.rc("axes", unicode_minus=False)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

DATA_DIR = Path("/content/drive/MyDrive/baram2026/open")
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"

TARGET_COLS = ["kpx_group_1", "kpx_group_2", "kpx_group_3"]
CAPACITY_KWH = {"kpx_group_1": 21600, "kpx_group_2": 21600, "kpx_group_3": 21000}

In [4]:
train_labels = pd.read_csv(TRAIN_DIR / "train_labels.csv", encoding="utf-8-sig")
train_labels["kst_dtm"] = pd.to_datetime(train_labels["kst_dtm"])

sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv", encoding="utf-8-sig")
sample_submission["forecast_kst_dtm"] = pd.to_datetime(sample_submission["forecast_kst_dtm"])

ldaps_train = pd.read_csv(TRAIN_DIR / "ldaps_train.csv", encoding="utf-8-sig")
gfs_train = pd.read_csv(TRAIN_DIR / "gfs_train.csv", encoding="utf-8-sig")
ldaps_test = pd.read_csv(TEST_DIR / "ldaps_test.csv", encoding="utf-8-sig")
gfs_test = pd.read_csv(TEST_DIR / "gfs_test.csv", encoding="utf-8-sig")

print("ldaps_train:", ldaps_train.shape, " gfs_train:", gfs_train.shape)
print("ldaps_test:", ldaps_test.shape, " gfs_test:", gfs_test.shape)

ldaps_train: (420864, 35)  gfs_train: (236736, 40)
ldaps_test: (140160, 35)  gfs_test: (78840, 40)


In [5]:
def aggregate_weather(df, prefix):
    df = df.copy()
    df["forecast_kst_dtm"] = pd.to_datetime(df["forecast_kst_dtm"])
    df["data_available_kst_dtm"] = pd.to_datetime(df["data_available_kst_dtm"])

    drop_cols = {"data_available_kst_dtm", "grid_id", "latitude", "longitude"}
    value_cols = [c for c in df.columns if c not in {"forecast_kst_dtm", *drop_cols}]

    agg_mean = df.groupby("forecast_kst_dtm")[value_cols].mean()
    agg_mean.columns = [f"{prefix}_{c}_mean" for c in agg_mean.columns]

    agg_std = df.groupby("forecast_kst_dtm")[value_cols].std()
    agg_std.columns = [f"{prefix}_{c}_std" for c in agg_std.columns]

    lead = df.groupby("forecast_kst_dtm").agg(data_available_kst_dtm=("data_available_kst_dtm", "first"))
    lead[f"{prefix}_lead_hours"] = (lead.index - lead["data_available_kst_dtm"]).dt.total_seconds() / 3600
    lead = lead[[f"{prefix}_lead_hours"]]

    return agg_mean.join(agg_std).join(lead).reset_index()


ldaps_train_agg = aggregate_weather(ldaps_train, "ldaps")
gfs_train_agg = aggregate_weather(gfs_train, "gfs")
ldaps_test_agg = aggregate_weather(ldaps_test, "ldaps")
gfs_test_agg = aggregate_weather(gfs_test, "gfs")

print(ldaps_train_agg.shape, gfs_train_agg.shape)

(26304, 62) (26304, 72)


In [6]:
train_weather = ldaps_train_agg.merge(gfs_train_agg, on="forecast_kst_dtm", how="inner")
test_weather = ldaps_test_agg.merge(gfs_test_agg, on="forecast_kst_dtm", how="inner")

train_df = train_labels.rename(columns={"kst_dtm": "forecast_kst_dtm"}).merge(
    train_weather, on="forecast_kst_dtm", how="left"
)
test_df = sample_submission[["forecast_id", "forecast_kst_dtm"]].merge(
    test_weather, on="forecast_kst_dtm", how="left"
)

print("train_df:", train_df.shape, " test_df:", test_df.shape)
print("결측 확인:", train_df.filter(like="_mean").isna().mean().max())

train_df: (26304, 136)  test_df: (8760, 134)
결측 확인: 0.0


In [7]:
def add_wind_speed_features(df):
    df = df.copy()
    pairs = {
        "gfs_ws_10m": ("gfs_heightAboveGround_10_10u_mean", "gfs_heightAboveGround_10_10v_mean"),
        "gfs_ws_80m": ("gfs_heightAboveGround_80_u_mean", "gfs_heightAboveGround_80_v_mean"),
        "gfs_ws_100m": ("gfs_heightAboveGround_100_100u_mean", "gfs_heightAboveGround_100_100v_mean"),
        "ldaps_ws_10m": ("ldaps_heightAboveGround_10_10u_mean", "ldaps_heightAboveGround_10_10v_mean"),
    }
    for new_col, (u_col, v_col) in pairs.items():
        if u_col in df.columns and v_col in df.columns:
            df[new_col] = np.sqrt(df[u_col] ** 2 + df[v_col] ** 2)

    # LDAPS 50m은 max/min 형태라 정확한 크기는 아니지만, 근사치로 추가 (허브높이 근접 고도)
    if {"ldaps_heightAboveGround_50_50MUmax_mean", "ldaps_heightAboveGround_50_50MVmax_mean"}.issubset(df.columns):
        df["ldaps_ws_50m_max_approx"] = np.sqrt(
            df["ldaps_heightAboveGround_50_50MUmax_mean"] ** 2 + df["ldaps_heightAboveGround_50_50MVmax_mean"] ** 2
        )
    return df


train_df = add_wind_speed_features(train_df)
test_df = add_wind_speed_features(test_df)

print([c for c in train_df.columns if "_ws_" in c])

['gfs_ws_10m', 'gfs_ws_80m', 'gfs_ws_100m', 'ldaps_ws_10m', 'ldaps_ws_50m_max_approx']


In [8]:
def add_disagreement_features(df):
    df = df.copy()
    if "ldaps_ws_10m" in df.columns and "gfs_ws_10m" in df.columns:
        df["ws_diff_ldaps_gfs"] = df["ldaps_ws_10m"] - df["gfs_ws_10m"]
        df["ws_diff_ldaps_gfs_abs"] = df["ws_diff_ldaps_gfs"].abs()
    return df


train_df = add_disagreement_features(train_df)
test_df = add_disagreement_features(test_df)

In [9]:
def add_air_density_feature(df):
    df = df.copy()
    if {"ldaps_surface_0_sp_mean", "ldaps_heightAboveGround_2_t_mean"}.issubset(df.columns):
        df["air_density"] = df["ldaps_surface_0_sp_mean"] / (
            287.05 * df["ldaps_heightAboveGround_2_t_mean"]
        )
    return df


train_df = add_air_density_feature(train_df)
test_df = add_air_density_feature(test_df)

print(train_df["air_density"].describe())

count    26304.000000
mean         1.125129
std          0.044788
min          1.037393
25%          1.085324
50%          1.120733
75%          1.162556
max          1.257640
Name: air_density, dtype: float64


In [10]:
def fit_power_curve(wind_speed, capacity_factor, n_bins=25):
    mask = wind_speed.notna() & capacity_factor.notna()
    ws, cf = wind_speed[mask].to_numpy(), capacity_factor[mask].to_numpy()

    bins = np.linspace(ws.min(), ws.max(), n_bins + 1)
    bin_idx = np.digitize(ws, bins[1:-1])
    centers = 0.5 * (bins[:-1] + bins[1:])
    means = np.array([
        cf[bin_idx == i].mean() if (bin_idx == i).any() else np.nan
        for i in range(n_bins)
    ])
    valid = ~np.isnan(means)
    means_filled = np.interp(centers, centers[valid], means[valid])
    return centers, means_filled


def apply_power_curve(wind_speed, centers, means):
    return np.interp(wind_speed, centers, means, left=means[0], right=means[-1])


best_ws_col = "ldaps_ws_10m"

power_curve_lookup = {}
for target in TARGET_COLS:
    cf = train_df[target] / CAPACITY_KWH[target]
    centers, means = fit_power_curve(train_df[best_ws_col], cf)
    power_curve_lookup[target] = (centers, means)

    feat_name = f"powercurve_{target}"
    train_df[feat_name] = apply_power_curve(train_df[best_ws_col], centers, means)
    test_df[feat_name] = apply_power_curve(test_df[best_ws_col], centers, means)

print("파워커브 feature 생성 완료:", [f"powercurve_{t}" for t in TARGET_COLS])

파워커브 feature 생성 완료: ['powercurve_kpx_group_1', 'powercurve_kpx_group_2', 'powercurve_kpx_group_3']


In [11]:
def add_calendar_features(df):
    df = df.copy()
    dt = df["forecast_kst_dtm"]
    df["hour"] = dt.dt.hour
    df["month"] = dt.dt.month
    df["dayofweek"] = dt.dt.dayofweek
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    return df


train_df = add_calendar_features(train_df)
test_df = add_calendar_features(test_df)

In [12]:
vrate_col = "gfs_planetaryBoundaryLayer_0_VRATE_mean"
if vrate_col in train_df.columns:
    for target in TARGET_COLS:
        sub_t = train_df.dropna(subset=[target, vrate_col])
        corr = sub_t[[vrate_col, target]].corr().iloc[0, 1]
        print(f"{target} vs VRATE: corr={corr:.3f}")
    corr_ws = train_df[[vrate_col, "ldaps_ws_10m"]].corr().iloc[0, 1]
    print("VRATE vs LDAPS 10m 풍속 상관(중복도):", round(corr_ws, 3))

kpx_group_1 vs VRATE: corr=0.376
kpx_group_2 vs VRATE: corr=0.383
kpx_group_3 vs VRATE: corr=0.393
VRATE vs LDAPS 10m 풍속 상관(중복도): 0.67


In [13]:
exclude_cols = {"forecast_id", "forecast_kst_dtm", *TARGET_COLS}
feature_cols = [c for c in train_df.columns if c not in exclude_cols]
feature_cols = [c for c in feature_cols if c in test_df.columns]

print("최종 feature 개수:", len(feature_cols))

최종 feature 개수: 150


In [14]:
PROCESSED_DIR = Path("/content/drive/MyDrive/baram2026/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

train_df.to_parquet(PROCESSED_DIR / "train_features_v1.parquet", index=False)
test_df.to_parquet(PROCESSED_DIR / "test_features_v1.parquet", index=False)

import json
with open(PROCESSED_DIR / "feature_cols_v1.json", "w") as f:
    json.dump(feature_cols, f, ensure_ascii=False, indent=2)

print("저장 완료:", PROCESSED_DIR)

저장 완료: /content/drive/MyDrive/baram2026/processed
